# TensorFlow Serving for Model Deployment
## AIAT 122 – Deep Learning

## Learning objectives
- Save a Keras model in SavedModel format for serving.
- Understand how TensorFlow Serving is used in production (REST/gRPC).
- Run local inference from the saved model (no Docker required in this notebook).

**Where is this used in real life?** Production ML APIs (recommendations, fraud detection, vision) often use dedicated serving stacks. **We use TensorFlow Serving (TFS) to serve models at scale** instead of loading the model inside the same app because TFS handles batching, versioning, and efficient inference; loading in-process does not scale across many requests.

**Prerequisites:** Basic Python, TensorFlow. If TensorFlow import fails, see DOCS/COLAB_SETUP.md.

**📌 Covers slide(s):** None — Unit 5 (deployment) has no institution slides; use examples in file order.


## Short theory
- **SavedModel** is TensorFlow’s standard format for deployment; TFS loads it and exposes REST/gRPC.
- **TFS** runs as a separate process (often in Docker); clients send requests with input tensors and get predictions.
- **REST API:** `POST /v1/models/<name>:predict` with a JSON body `{"instances": [...]}`.
- **Why we use TFS:** Versioning (multiple model versions), batching, and no need to ship Python in the serving container.

## Inputs & Outputs
**Inputs:** TensorFlow, a small Keras model (built here), and a save path.  
**Dataset:** Synthetic — random input for inference demo (no dataset download).  
**Outputs:** Saved SavedModel directory, local inference from the saved model, and TFS setup instructions (for use with Docker). Run time: under ~2 min.


In [5]:
%pip install tensorflow tensorflow-serving-api -q
import numpy as np
try:
    import tensorflow as tf
    print(f'TensorFlow version: {tf.__version__}')
    print('✅ Setup complete!')
except Exception as e:
    err = str(e).lower()
    if "charset_normalizer" in err or "md__mypyc" in err or "partially initialized" in err:
        print('⚠️ Fix: pip install --upgrade charset-normalizer requests, then restart kernel.')
        raise RuntimeError('Fix: pip install --upgrade charset-normalizer requests, then restart kernel.') from e
    raise

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cryptography 46.0.3 requires typing-extensions>=4.13.2; python_full_version < "3.11", but you have typing-extensions 4.5.0 which is incompatible.
pydantic 2.10.6 requires typing-extensions>=4.12.2, but you have typing-extensions 4.5.0 which is incompatible.
pydantic-core 2.27.2 requires typing-extensions!=4.7.0,>=4.6.0, but you have typing-extensions 4.5.0 which is incompatible.
torch 2.4.1 requires typing-extensions>=4.8.0, but you have typing-extensions 4.5.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
TensorFlow version: 2.13.0
✅ Setup complete!


## Part 1: Save Model in SavedModel Format


In [6]:
# Create a simple model for demonstration
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(10,)),
 tf.keras.layers.Dense(32, activation='relu'),
 tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Save in SavedModel format
model_path = './saved_model'
model.save(model_path, save_format='tf')
print(f'✅ Model saved to {model_path}')

INFO:tensorflow:Assets written to: ./saved_model/assets


INFO:tensorflow:Assets written to: ./saved_model/assets


✅ Model saved to ./saved_model


## Part 2: TensorFlow Serving Setup

**Note**: Full TensorFlow Serving requires Docker. Here we demonstrate the concept.

In [7]:
print('📦 TensorFlow Serving Setup:')
print('\n1. Install TensorFlow Serving:')
print(' docker pull tensorflow/serving')
print('\n2. Start serving container:')
print(' docker run -p 8501:8501 --mount type=bind,source=/path/to/model,target=/models/model -e MODEL_NAME=model tensorflow/serving')
print('\n3. Test REST API:')
print(' curl -d \'{"instances": [[1,2,3,...]]}\' -X POST http://localhost:8501/v1/models/model:predict')
print('\n✅ Serving setup understood!')

📦 TensorFlow Serving Setup:

1. Install TensorFlow Serving:
 docker pull tensorflow/serving

2. Start serving container:
 docker run -p 8501:8501 --mount type=bind,source=/path/to/model,target=/models/model -e MODEL_NAME=model tensorflow/serving

3. Test REST API:
 curl -d '{"instances": [[1,2,3,...]]}' -X POST http://localhost:8501/v1/models/model:predict

✅ Serving setup understood!


## Part 3: REST API Client Example


In [8]:
# Run inference using the saved model (no TFS server needed for this step)
loaded = tf.saved_model.load(model_path)
infer = loaded.signatures["serving_default"]
# Example: one batch of shape (1, 10)
x = tf.constant(np.random.randn(1, 10).astype(np.float32))
out = infer(x)
# Output key varies by TF/Keras version (e.g. "output_0" or layer name); use first output to avoid KeyError
pred = list(out.values())[0].numpy()
print("Sample input shape:", x.shape)
print("Prediction (sigmoid output):", pred)
print("\n✅ Local inference from SavedModel works! In production you'd run TFS and call this via REST.")



Sample input shape: (1, 10)
Prediction (sigmoid output): [[0.5179364]]

✅ Local inference from SavedModel works! In production you'd run TFS and call this via REST.


## 🧩 Mini-exercise

**Try it:** Change the model (e.g. add one more Dense layer), save to a new SavedModel directory, and run inference from the new path. Compare output shape with the original.

---

## Summary
**What you did**
- Built a small Keras model and saved it as SavedModel.
- Ran local inference from the saved model.
- Saw how to run TensorFlow Serving in Docker and call the REST API.

**In real life you'd also:** Run TFS in Docker/Kubernetes, use gRPC for low latency, and add monitoring and A/B tests for model versions.

**The main idea:** SavedModel is the deployment format; TensorFlow Serving serves it at scale via REST/gRPC.

**Next:** `03_onnx_conversion.ipynb` shows how to export models to ONNX for cross-platform deployment.